# Transformers

In [15]:
import os
import json
import torch
import pandas as pd
from typing import Any, Dict
from datetime import datetime
from dataclasses import dataclass
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader

# Create the results directory
results_dir = r'/data/results/transformers'
os.makedirs(results_dir, exist_ok=True)
print(f"Results will be saved to: {results_dir}")

Results will be saved to: /data/results/transformers


In [16]:
class ArxivDataset(Dataset):
    def __init__(self, pt_tensor):
        self.input_ids = pt_tensor['input_ids']
        self.token_type_ids = pt_tensor['token_type_ids']
        self.attention_mask = pt_tensor['attention_mask']
        self.labels = pt_tensor['labels']

        self.length = len(self.input_ids)

    def __getitem__(self, idx):  # returns ONE dict
        return {
            'input_ids': self.input_ids[idx],
            'token_type_ids': self.token_type_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels[idx]
        }

    def __len__(self):
        return self.length


@dataclass
class Configuration:
    model_name: str
    output_dir: str
    tokenizer: Any

    task: str = 'text-classification'

    num_labels: int = 40
    batch_size: int = 16
    learning_rate: float = 2e-5
    num_epochs: int = 3
    seed: int = 42

    eval_metric: str = 'macro_f1'
    eval_steps: int = 500

    model_device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    num_workers: int = 2 if torch.cuda.is_available() else 0

    def tokenize(self, d) -> Dict[str, Any]:
        tokens = self.tokenizer(
            d['text'].tolist(),
            padding="max_length",
            truncation=True,
            max_length=256,
            return_tensors='pt'
        )

        tokens["labels"] = torch.tensor(d["label"].values, dtype=torch.long)
        tokens = {k: v.to(self.model_device) for k, v in tokens.items()}

        return tokens

In [17]:
sci_config = Configuration(
    model_name='allenai/scibert_scivocab_uncased',
    output_dir=f'{results_dir}/scibert_scivocab_uncased',
    tokenizer=AutoTokenizer.from_pretrained('allenai/scibert_scivocab_uncased')
)
bert_config = Configuration(
    model_name='google-bert/bert-base-uncased',
    output_dir=f'{results_dir}/bert_base_uncased',
    tokenizer=AutoTokenizer.from_pretrained('google-bert/bert-base-uncased')
)

In [18]:
print("Splitting data: using val in training data for final training")
data = pd.read_parquet('../data/processed/arxiv_text.parquet')
data['text'] = data.text.str.lower()

print("Tokenizing title+abstract for Sci Bert")
sci_bert_data = sci_config.tokenize(data)

print("Tokenizing title+abstract for Bert")
bert_data = bert_config.tokenize(data)

print('\nTokenizing completed!')

Splitting data: using val in training data for final training
Tokenizing title+abstract for Sci Bert
Tokenizing title+abstract for Bert

Tokenizing completed!


In [19]:
bert_results = []
sci_bert_results = []

criterion = torch.nn.CrossEntropyLoss()


def train(example, model, optimizer):
    outputs = model(**example)
    l = criterion(outputs.logits, example['labels'])
    l.backward()
    optimizer.step()
    optimizer.zero_grad()
    return l.item()


def full_training_loop(name, d, model, config, adam):
    res = {'name': name}
    ds = ArxivDataset(d)
    dl = DataLoader(ds, batch_size=32, shuffle=True, num_workers=0)

    losses = []
    for epoch in range(config.num_epochs):
        print(f"Epoch: {epoch + 1}")
        model.train()
        running_loss = 0
        batch_total = 0
        for batch in dl:
            batch_total += len(batch)
            batch = {k: v.to(config.model_device) for k, v in batch.items()}
            running_loss += train(batch, model, adam)
            if batch_total % 1000 == 0:
                print(f"Epoch: {epoch} - total batches: {batch_total} - running loss: {running_loss}")
        losses.append(running_loss / len(dl))
        print(f"Epoch: {epoch} - average loss: {running_loss}")

    res['losses'] = losses
    return res

## 2. BERT
### 2.1 BERT Frozen

In [20]:
bert_frozen_name = 'bert_frozen'

bert = AutoModelForSequenceClassification.from_pretrained(
    bert_config.model_name,
    num_labels=bert_config.num_labels,
)

bert.to(bert_config.model_device)

# Freeze BERT parameters
for param in bert.bert.parameters():
    param.requires_grad = False

adamO = torch.optim.AdamW(
    params=filter(lambda p: p.requires_grad, bert.parameters()),
    lr=bert_config.learning_rate
)

bert_results.append(full_training_loop(bert_frozen_name, bert_data, bert, bert_config, adamO))

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch: 1
Epoch: 0 - average loss: 15455.336267471313
Epoch: 2
Epoch: 1 - average loss: 14598.14949131012
Epoch: 3
Epoch: 2 - average loss: 14081.820662975311


### 2.2 BERT Fine-tuned

In [21]:
bert_ft_name = 'bert_fine_tuned'

bert = AutoModelForSequenceClassification.from_pretrained(
    bert_config.model_name,
    num_labels=bert_config.num_labels,
)

bert.to(bert_config.model_device)

adamO = torch.optim.AdamW(
    params=bert.parameters(),
    lr=bert_config.learning_rate
)

bert_results.append(full_training_loop(bert_ft_name, bert_data, bert, bert_config, adamO))

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


KeyError: 'input_ids'

## 3. SciBERT
### 3.1 SciBERT Frozen

In [ ]:
sci_bert_frozen_name = 'sci_bert_frozen'

sci_bert = AutoModelForSequenceClassification.from_pretrained(
    sci_config.model_name,
    num_labels=sci_config.num_labels,
)

sci_bert.to(sci_config.model_device)

# Freeze BERT parameters
for param in sci_bert.bert.parameters():
    param.requires_grad = False

adamO = torch.optim.AdamW(
    params=filter(lambda p: p.requires_grad, sci_bert.parameters()),
    lr=sci_config.learning_rate
)

sci_bert_results.append(full_training_loop(sci_bert_frozen_name, sci_bert_data, sci_bert, sci_config, adamO))

### 3.2 SciBERT Fine-tuned

In [ ]:
sci_bert_frozen_name = 'sci_bert_fine_tuned'

sci_bert = AutoModelForSequenceClassification.from_pretrained(
    sci_config.model_name,
    num_labels=sci_config.num_labels,
)

sci_bert.to(sci_config.model_device)

adamO = torch.optim.AdamW(
    params=sci_bert.parameters(),
    lr=sci_config.learning_rate
)

sci_bert_results.append(full_training_loop(sci_bert_frozen_name, sci_bert_data, sci_bert, sci_config, adamO))

## 4. Save Data

In [ ]:
bert_dump = {
    "timestamp": datetime.now().strftime("%Y%m%d_%H%M%S"),
    "experiment_type": "transformers",
    "pretrained_model": bert_config.model_name,
    "results": bert_results
}

with open(bert_config.output_dir, 'w') as f:
    json.dump(bert_dump, f, indent=2)

print(f"Results saved to {bert_config.output_dir}")


sci_bert_dump = {
    "timestamp": datetime.now().strftime("%Y%m%d_%H%M%S"),
    "experiment_type": "transformers",
    "pretrained_model": sci_config.model_name,
    "results": sci_bert_results
}

with open(bert_config.output_dir, 'w') as f:
    json.dump(sci_bert_dump, f, indent=2)

print(f"Results saved to {sci_config.output_dir}")